# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the `FAIR^2` dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}")

print("\nOther metadata properties:")
print(f"Authors: {getattr(metadata, 'author', [])}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and describe fields using @id
record_sets = list(dataset.record_sets.keys())
print(f"RecordSets found: {record_sets}\n")
for rs_id in record_sets:
    record_set = dataset.record_sets[rs_id]
    print(f"RecordSet @id: {rs_id}")
    print(f"  Name: {getattr(record_set, 'name', rs_id)}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    
    # List field @ids
    field_ids = list(record_set.fields.keys()) if hasattr(record_set, 'fields') else []
    print(f"  Field @ids: {field_ids}")
    for fid in field_ids:
        field = record_set.fields[fid]
        print(f"    Field {fid} - Name: {getattr(field, 'name', '')}, DataType: {getattr(field, 'dataType', '')}")
    print("")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set and store in DataFrames
dfs = {}
record_set_ids = list(dataset.record_sets.keys())

for rs_id in record_set_ids:
    print(f"Loading data for RecordSet {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    if len(records) == 0:
        print(f"  [Warning] No records loaded for {rs_id}, moving on.\n")
        continue
    df = pd.DataFrame(records)
    dfs[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Shape: {df.shape}\n")

# Pick a primary record set (usually the first or main table)
if dfs:
    main_record_set_id = list(dfs.keys())[0]
    main_df = dfs[main_record_set_id]
    print(f"First few rows of RecordSet {main_record_set_id}:")
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
# Example EDA: filter by a numeric column, normalize, and group
import numpy as np

# If the data & fields are loaded, inspect numeric columns
if dfs:
    df = main_df.copy()
    print(f"All columns: {df.columns.tolist()}")
    
    # Try to find a likely numeric field
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Try to cast columns containing numeric values
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # select the first numeric field
        print(f"Using numeric field '{numeric_field}' for filtering and normalization.")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} entries")
        
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try grouping by a string field if available
        group_candidates = df.select_dtypes(include=[object]).columns.tolist()
        group_candidates = [c for c in group_candidates if c != numeric_field]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            print(grouped.head())
    else:
        print("No numeric fields found in the DataFrame for EDA.")
else:
    print("No DataFrames loaded to perform EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field if available
if dfs and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    # If grouping field available, plot boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR<sup>2</sup> dataset via its Croissant schema, inspected its structure, extracted records, and performed initial exploratory analysis. This process can be extended to deeper domain-specific investigations, feature engineering, or training machine learning models depending on downstream tasks.